In [2]:
import rpy2  ##THE IMPORTANT RESULTS ARE GIVEN BY THE CODE IN LINE 38 AND BELOW, BUT SOME OF THE ABOVE MIGHT BE NECESSARY TO RUN BEFORE
print(rpy2.__version__)

3.5.11


In [3]:
%load_ext rpy2.ipython

In [4]:
%%R

# Load libraries
library(DESeq2)

# Load the count data and column data
counts_DEseq <- read.table("../HP_counts.txt", header= TRUE, sep="\t", row.names=1 ) 

coldata_DEseq <- read.table("../metadata_clean.txt", header= TRUE, sep=",", row.names=1) 
coldata_DEseq$library_name <- gsub("\\d+", "", coldata_DEseq$library_name)
coldata_DEseq$library_name <- factor(coldata_DEseq$library_name, levels = c("Hp-", "Gast", "Atr", "EA", "Met"))

#want rownames of coldata to match column names of counts
all(rownames(coldata_DEseq) == colnames(counts_DEseq))   #the files are located one step above in the directory
#coldata_DEseq

R[write to console]: Loading required package: S4Vectors

R[write to console]: Loading required package: stats4

R[write to console]: Loading required package: BiocGenerics

R[write to console]: 
Attaching package: 'BiocGenerics'


R[write to console]: The following objects are masked from 'package:stats':

    IQR, mad, sd, var, xtabs


R[write to console]: The following objects are masked from 'package:base':

    Filter, Find, Map, Position, Reduce, anyDuplicated, aperm, append,
    as.data.frame, basename, cbind, colnames, dirname, do.call,
    duplicated, eval, evalq, get, grep, grepl, intersect, is.unsorted,
    lapply, mapply, match, mget, order, paste, pmax, pmax.int, pmin,
    pmin.int, rank, rbind, rownames, sapply, setdiff, sort, table,
    tapply, union, unique, unsplit, which.max, which.min


R[write to console]: 
Attaching package: 'S4Vectors'


R[write to console]: The following object is masked from 'package:utils':

    findMatches


R[write to console]: The following 

[1] TRUE


In [5]:
%%R

dds <- DESeqDataSetFromMatrix(countData=counts_DEseq, 
                                      colData=coldata_DEseq, 
                                      design=~library_name) 
dds <- DESeq(dds)


R[write to console]:   Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

R[write to console]: estimating size factors

R[write to console]:   Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]:   Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) 

In [6]:
%%R
#Compare with the healthy - should not have bacterial reads for the healty so should get significant results (none vs bacterial expression)
res_Hp_vs_Gast <- results(dds, contrast = c("library_name", "Hp-", "Gast"))
#summary(res_Hp_vs_Gast)
res_Hp_vs_Gast

log2 fold change (MLE): library_name Hp- vs Gast 
Wald test p-value: library_name Hp- vs Gast 
DataFrame with 1609 rows and 6 columns
           baseMean log2FoldChange     lfcSE      stat     pvalue      padj
          <numeric>      <numeric> <numeric> <numeric>  <numeric> <numeric>
HP_0573   1.1160388        3.33421   1.73115   1.92600 0.05410375 0.0986472
HP_0020   1.5343544        3.96808   1.41181   2.81064 0.00494434 0.0593805
HP_1325   3.5381085        1.90596   1.23955   1.53762 0.12414233 0.1624726
HP_0038   0.2458265        6.58250   4.15047   1.58597 0.11274710 0.1515056
HP_0765   0.0023665        0.00000   4.92647   0.00000 1.00000000        NA
...             ...            ...       ...       ...        ...       ...
HP_t07  7.39309e+00      1.2002482  1.252559  0.958237   0.337943  0.368310
HP_t08  3.31763e-02      7.8927053  4.918341  1.604749   0.108549        NA
HP_r07  5.80328e+03     -0.0584964  0.158301 -0.369526   0.711736  0.732669
HP_t12  2.51393e-02      7.892

In [7]:
%%R
# ordered (increasing p-adjusted values)
res_Hp_vs_Gast_ordered <- order(res_Hp_vs_Gast$padj, decreasing = FALSE, na.last=TRUE)
res_Hp_vs_Gast_ordered_decreasing = res_Hp_vs_Gast[res_Hp_vs_Gast_ordered,]
res_Hp_vs_Gast_ordered_decreasing

log2 fold change (MLE): library_name Hp- vs Gast 
Wald test p-value: library_name Hp- vs Gast 
DataFrame with 1609 rows and 6 columns
         baseMean log2FoldChange     lfcSE      stat      pvalue        padj
        <numeric>      <numeric> <numeric> <numeric>   <numeric>   <numeric>
HP_0211   6.41533        7.58620  1.533075   4.94835 7.48435e-07 0.000916833
HP_0068  19.04025        4.13870  0.896853   4.61469 3.93679e-06 0.001832091
HP_0476  11.95066        5.10635  1.113111   4.58746 4.48675e-06 0.001832091
HP_1326  13.44533        6.18280  1.408953   4.38822 1.14280e-05 0.003499834
HP_0226  11.59063        5.74439  1.326644   4.33001 1.49101e-05 0.003652965
...           ...            ...       ...       ...         ...         ...
HP_t27  0.0000000             NA        NA        NA          NA          NA
HP_t01  0.0364315        8.12416   4.92137   1.65079   0.0987807          NA
HP_t08  0.0331763        7.89271   4.91834   1.60475   0.1085490          NA
HP_t12  0.0251393  

In [8]:
%%R
#do the same for all the cancer stages
res_Hp_vs_Atr <- results(dds, contrast = c("library_name", "Hp-", "Atr"))
res_Hp_vs_Atr

log2 fold change (MLE): library_name Hp- vs Atr 
Wald test p-value: library_name Hp- vs Atr 
DataFrame with 1609 rows and 6 columns
           baseMean log2FoldChange     lfcSE      stat    pvalue      padj
          <numeric>      <numeric> <numeric> <numeric> <numeric> <numeric>
HP_0573   1.1160388        3.77419   1.64095   2.30000 0.0214480 0.0676120
HP_0020   1.5343544        2.89170   1.32955   2.17495 0.0296342 0.0717016
HP_1325   3.5381085        1.82371   1.19338   1.52819 0.1264664 0.1588999
HP_0038   0.2458265        5.28111   3.82046   1.38232 0.1668721 0.1972125
HP_0765   0.0023665        7.73583   4.54205   1.70316 0.0885386        NA
...             ...            ...       ...       ...       ...       ...
HP_t07  7.39309e+00      1.7952021  1.208681   1.48526 0.1374758  0.168855
HP_t08  3.31763e-02      6.6637108  4.534153   1.46967 0.1416510        NA
HP_r07  5.80328e+03     -0.0245023  0.149477  -0.16392 0.8697942  0.880524
HP_t12  2.51393e-02      7.7358715  4.54205

In [9]:
%%R 
res_Hp_vs_EA <- results(dds, contrast = c("library_name", "Hp-", "EA"))
res_Hp_vs_EA

log2 fold change (MLE): library_name Hp- vs EA 
Wald test p-value: library_name Hp- vs EA 
DataFrame with 1609 rows and 6 columns
           baseMean log2FoldChange     lfcSE      stat     pvalue      padj
          <numeric>      <numeric> <numeric> <numeric>  <numeric> <numeric>
HP_0573   1.1160388        4.00471   1.72873   2.31655 0.02052809 0.0995208
HP_0020   1.5343544        3.81600   1.39300   2.73941 0.00615499 0.0905585
HP_1325   3.5381085        2.60992   1.23850   2.10732 0.03508944 0.1025010
HP_0038   0.2458265        7.73662   4.16478   1.85763 0.06322187        NA
HP_0765   0.0023665        0.00000   4.93025   0.00000 1.00000000        NA
...             ...            ...       ...       ...        ...       ...
HP_t07  7.39309e+00      0.6836697  1.248011  0.547807   0.583824  0.612110
HP_t08  3.31763e-02      0.0000000  4.930247  0.000000   1.000000        NA
HP_r07  5.80328e+03      0.0793832  0.158272  0.501562   0.615976  0.642086
HP_t12  2.51393e-02      7.1989500

In [10]:
%%R
res_Hp_vs_Met <- results(dds, contrast = c("library_name", "Hp-", "Met"))
res_Hp_vs_Met

log2 fold change (MLE): library_name Hp- vs Met 
Wald test p-value: library_name Hp- vs Met 
DataFrame with 1609 rows and 6 columns
           baseMean log2FoldChange     lfcSE      stat    pvalue      padj
          <numeric>      <numeric> <numeric> <numeric> <numeric> <numeric>
HP_0573   1.1160388        2.21746   1.96549  1.128196  0.259237  0.821315
HP_0020   1.5343544        1.53095   1.57986  0.969044  0.332523  0.821315
HP_1325   3.5381085        1.07232   1.40506  0.763183  0.445354  0.824687
HP_0038   0.2458265        0.00000   4.58562  0.000000  1.000000  1.000000
HP_0765   0.0023665        0.00000   5.42657  0.000000  1.000000  1.000000
...             ...            ...       ...       ...       ...       ...
HP_t07  7.39309e+00     -0.3912532   1.41522 -0.276461  0.782194         1
HP_t08  3.31763e-02      0.0000000   5.42657  0.000000  1.000000         1
HP_r07  5.80328e+03      0.0470845   0.17698  0.266044  0.790206         1
HP_t12  2.51393e-02      0.0000000   5.4265

In [10]:
%%R
res=results(dds)

In [12]:
%%R

#Do the same analysis, but remove all genes that have a low expression
# Read the data into a data frame
data <- read.csv("../HP_counts_fixed_163.txt", header = TRUE, sep = "\t")
#data2 <- read.csv("../bacteria_counts_clean_highexpression.txt", header = TRUE, sep = ”,")

In [ ]:
%%R
# Function to check if at least 4 columns have a value over 10
check_high_expression <- function(row) {
  sum(as.numeric(row[-1]) > 10) >= 4
}

In [ ]:

%%R
# Apply the function to each row and filter the data
filtered_indices <- apply(data, 1, check_high_expression)
filtered_data <- data[c(TRUE, filtered_indices), ] # Keep the header and rows with high expression

# Write the filtered data to a new file
#write.csv(filtered_data, "../HP_counts_fixed_163_highexpression.txt", row.names = FALSE)


In [11]:
%%R 
#use the file from Julias notebook (the function above stopped working for some wierd reason)
highex_counts_DEseq <- read.csv("../HP_counts_fixed_163_highexpression_withrows", header= TRUE, sep=",", row.names=1 ) 


In [12]:
%%R 

highex_coldata_DEseq <- read.table("../metadata_clean.txt", header= TRUE, sep=",", row.names=1) 
highex_coldata_DEseq$library_name <- gsub("\\d+", "", coldata_DEseq$library_name)
highex_coldata_DEseq$library_name <- factor(coldata_DEseq$library_name, levels = c("Hp-", "Gast", "Atr", "EA", "Met"))

#want rownames of coldata to match column names of counts
all(rownames(highex_coldata_DEseq) == colnames(highex_counts_DEseq))   #the files are located one step above in the directory


[1] TRUE


In [13]:
%%R
highex_dds <- DESeqDataSetFromMatrix(countData=highex_counts_DEseq, 
                                      colData=highex_coldata_DEseq , 
                                      design=~library_name) 
highex_dds <- DESeq(highex_dds)

R[write to console]:   Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

R[write to console]: estimating size factors

R[write to console]:   Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]:   Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) 

In [14]:
%%R
highex_res_Hp_vs_Gast <- results(highex_dds, contrast = c("library_name", "Hp-", "Gast"))
#summary(res_Hp_vs_Gast)
highex_res_Hp_vs_Gast

log2 fold change (MLE): library_name Hp- vs Gast 
Wald test p-value: library_name Hp- vs Gast 
DataFrame with 853 rows and 6 columns
          baseMean log2FoldChange     lfcSE      stat      pvalue       padj
         <numeric>      <numeric> <numeric> <numeric>   <numeric>  <numeric>
HP_0573    1.11604        3.36358   1.58700   2.11945 3.40521e-02 0.06347423
HP_0020    1.53435        3.95351   1.33546   2.96042 3.07224e-03 0.02809003
HP_1325    3.53811        1.90219   1.20561   1.57778 1.14616e-01 0.15097566
HP_0828    1.65759        3.82469   1.42739   2.67949 7.37345e-03 0.03541622
HP_1237    7.03845        5.36441   1.31485   4.07986 4.50625e-05 0.00547188
...            ...            ...       ...       ...         ...        ...
HP_r01  3386.31645     -0.2371620  0.354096 -0.669768   0.5030058  0.5390581
HP_r04  3383.35399      0.1013168  0.233517  0.433874   0.6643798  0.6963290
HP_r06    58.43071      2.0480272  0.881418  2.323558   0.0201492  0.0481401
HP_t07     7.39309  

In [15]:
%%R
# Assuming res_Hp_vs_Gast is your initial DESeq2 results data frame
# Remove rows with NA in the padj column
clean_res_Hp_vs_Gast <- highex_res_Hp_vs_Gast[!is.na(highex_res_Hp_vs_Gast$padj), ]

# Ordering by adjusted p-value (padj)
highex_res_Hp_vs_Gast_ordered <- order(clean_res_Hp_vs_Gast$padj, decreasing = FALSE)
highex_res_Hp_vs_Gast_ordered_decreasing <- clean_res_Hp_vs_Gast[highex_res_Hp_vs_Gast_ordered, ]

# Adding cut-off for padj < 0.05
sign_highex_res_Hp_vs_Gast <- highex_res_Hp_vs_Gast_ordered_decreasing[highex_res_Hp_vs_Gast_ordered_decreasing$padj <= 0.05, ]

write.csv(sign_highex_res_Hp_vs_Gast, "../DESeq-Sofia/significant_DEseq_results_Hp_vs_Gast.csv", row.names = TRUE)
# Display the significant results
sign_highex_res_Hp_vs_Gast


log2 fold change (MLE): library_name Hp- vs Gast 
Wald test p-value: library_name Hp- vs Gast 
DataFrame with 365 rows and 6 columns
         baseMean log2FoldChange     lfcSE      stat      pvalue       padj
        <numeric>      <numeric> <numeric> <numeric>   <numeric>  <numeric>
HP_1326  13.44533        6.16245  1.335016   4.61601 3.91187e-06 0.00124985
HP_0068  19.04025        4.13930  0.901611   4.59100 4.41124e-06 0.00124985
HP_0476  11.95066        5.09414  1.096769   4.64468 3.40611e-06 0.00124985
HP_0226  11.59063        5.70313  1.266982   4.50135 6.75219e-06 0.00143484
HP_1349   3.54459        5.61259  1.306494   4.29591 1.73975e-05 0.00295757
...           ...            ...       ...       ...         ...        ...
HP_1268  1.708965        3.04140   1.31219   2.31781   0.0204597  0.0482347
HP_0196  2.552595        3.08033   1.33017   2.31574   0.0205724  0.0483055
HP_0807  0.874437        5.27485   2.28293   2.31056   0.0208573  0.0488393
HP_1304  3.675834        2.7937

In [16]:
%%R

highex_res_Hp_vs_Atr <- results(highex_dds, contrast = c("library_name", "Hp-", "Atr"))
clean_res_Hp_vs_Atr <- highex_res_Hp_vs_Atr[!is.na(highex_res_Hp_vs_Atr$padj), ]

# Ordering by adjusted p-value (padj)
highex_res_Hp_vs_Atr_ordered <- order(clean_res_Hp_vs_Atr$padj, decreasing = FALSE)
highex_res_Hp_vs_Atr_ordered_decreasing <- clean_res_Hp_vs_Atr[highex_res_Hp_vs_Atr_ordered, ]

# Adding cut-off for padj < 0.05
sign_highex_res_Hp_vs_Atr <- highex_res_Hp_vs_Atr_ordered_decreasing[highex_res_Hp_vs_Atr_ordered_decreasing$padj <= 0.05, ]

write.csv(sign_highex_res_Hp_vs_Atr, "../DESeq-Sofia/significant_DEseq_results_Hp_vs_Atr.csv", row.names = TRUE)
# Display the significant results
sign_highex_res_Hp_vs_Atr

log2 fold change (MLE): library_name Hp- vs Atr 
Wald test p-value: library_name Hp- vs Atr 
DataFrame with 444 rows and 6 columns
         baseMean log2FoldChange     lfcSE      stat      pvalue       padj
        <numeric>      <numeric> <numeric> <numeric>   <numeric>  <numeric>
HP_0226  11.59063        5.48335  1.199523   4.57128 4.84757e-06 0.00328141
HP_0068  19.04025        3.82099  0.862717   4.42902 9.46613e-06 0.00328141
HP_0476  11.95066        4.58452  1.045424   4.38532 1.15814e-05 0.00328141
HP_0790   2.57512        5.85591  1.386940   4.22218 2.41949e-05 0.00514141
HP_1435   3.33729        5.02568  1.229822   4.08651 4.37915e-05 0.00744456
...           ...            ...       ...       ...         ...        ...
HP_0973  0.838867        3.55605   1.58522   2.24326   0.0248803  0.0480641
HP_1155  1.292483        2.96886   1.32644   2.23821   0.0252076  0.0485861
HP_0625  2.259920        2.75441   1.23496   2.23036   0.0257238  0.0494688
HP_0100  1.756684        3.12654 

In [17]:
%%R

highex_res_Hp_vs_EA <- results(highex_dds, contrast = c("library_name", "Hp-", "EA"))
clean_res_Hp_vs_EA <- highex_res_Hp_vs_EA[!is.na(highex_res_Hp_vs_EA$padj), ]

# Ordering by adjusted p-value (padj)
highex_res_Hp_vs_EA_ordered <- order(clean_res_Hp_vs_EA$padj, decreasing = FALSE)
highex_res_Hp_vs_EA_ordered_decreasing <- clean_res_Hp_vs_EA[highex_res_Hp_vs_EA_ordered, ]

# Adding cut-off for padj < 0.05
sign_highex_res_Hp_vs_EA <- highex_res_Hp_vs_EA_ordered_decreasing[highex_res_Hp_vs_EA_ordered_decreasing$padj <= 0.05, ]

write.csv(sign_highex_res_Hp_vs_EA, "../DESeq-Sofia/significant_DEseq_results_Hp_vs_EA.csv", row.names = TRUE)
# Display the significant results
sign_highex_res_Hp_vs_EA


log2 fold change (MLE): library_name Hp- vs EA 
Wald test p-value: library_name Hp- vs EA 
DataFrame with 47 rows and 6 columns
         baseMean log2FoldChange     lfcSE      stat      pvalue       padj
        <numeric>      <numeric> <numeric> <numeric>   <numeric>  <numeric>
HP_0226  11.59063        6.07661   1.25297   4.84977 1.23602e-06 0.00105062
HP_0219   4.94728        7.13571   1.59198   4.48229 7.38447e-06 0.00158773
HP_1326  13.44533        5.87867   1.31227   4.47979 7.47167e-06 0.00158773
HP_0476  11.95066        4.86339   1.07596   4.52004 6.18285e-06 0.00158773
HP_0068  19.04025        3.81805   0.89323   4.27443 1.91624e-05 0.00325761
...           ...            ...       ...       ...         ...        ...
HP_0668  1.430707        4.67381   1.51764   3.07966  0.00207235  0.0410531
HP_1340  6.089709        3.80161   1.24375   3.05656  0.00223890  0.0432515
HP_0293  0.556064        5.76238   1.88998   3.04891  0.00229675  0.0433830
HP_0866  1.832363        4.20086   1

In [18]:
%%R
highex_res_Hp_vs_Met <- results(highex_dds, contrast = c("library_name", "Hp-", "Met"))
clean_res_Hp_vs_Met <- highex_res_Hp_vs_Met[!is.na(highex_res_Hp_vs_Met$padj), ]

# Ordering by adjusted p-value (padj)
highex_res_Hp_vs_Met_ordered <- order(clean_res_Hp_vs_Met$padj, decreasing = FALSE)
highex_res_Hp_vs_Met_ordered_decreasing <- clean_res_Hp_vs_Met[highex_res_Hp_vs_Met_ordered, ]

write.csv(highex_res_Hp_vs_Met_ordered_decreasing, "../DESeq-Sofia/DEseq_results_Hp_vs_Met.csv", row.names = TRUE)

highex_res_Hp_vs_Met_ordered_decreasing

log2 fold change (MLE): library_name Hp- vs Met 
Wald test p-value: library_name Hp- vs Met 
DataFrame with 850 rows and 6 columns
           baseMean log2FoldChange     lfcSE        stat      pvalue      padj
          <numeric>      <numeric> <numeric>   <numeric>   <numeric> <numeric>
HP_0476    11.95066        4.45122   1.27024     3.50424 0.000457908  0.389222
HP_0573     1.11604        2.30690   1.80517     1.27794 0.201270813  0.436564
HP_0828     1.65759        2.52951   1.61802     1.56333 0.117974015  0.436564
HP_1237     7.03845        4.02537   1.50581     2.67323 0.007512485  0.436564
HP_0515     1.83949        2.46546   1.50195     1.64150 0.100693116  0.436564
...             ...            ...       ...         ...         ...       ...
HP_0127   16.055710     0.02240595  1.405570  0.01594084    0.987282         1
HP_1236    1.322057     0.00000000  1.786678  0.00000000    1.000000         1
HP_0111    5.413904     0.00728006  1.443209  0.00504436    0.995975         1


In [19]:
%%R

# Adding cut-off for padj < 0.05
sign_highex_res_Hp_vs_Met <- highex_res_Hp_vs_Met_ordered_decreasing[highex_res_Hp_vs_Met_ordered_decreasing$padj <= 0.05, ]

write.csv(sign_highex_res_Hp_vs_Met, "../DESeq-Sofia/significant_DEseq_results_Hp_vs_Met.csv", row.names = FALSE)
# Display the significant results
sign_highex_res_Hp_vs_Met


log2 fold change (MLE): library_name Hp- vs Met 
Wald test p-value: library_name Hp- vs Met 
DataFrame with 0 rows and 6 columns


In [20]:
%%R
#Now compare the first cancer stage to the last (Gast vs Met)

highex_res_Gast_vs_Met <- results(highex_dds, contrast = c("library_name", "Gast", "Met"))
clean_res_Gast_vs_Met <- highex_res_Gast_vs_Met[!is.na(highex_res_Gast_vs_Met$padj), ]

# Ordering by adjusted p-value (padj)
highex_res_Gast_vs_Met_ordered <- order(clean_res_Gast_vs_Met$padj, decreasing = FALSE)
highex_res_Gast_vs_Met_ordered_decreasing <- clean_res_Gast_vs_Met[highex_res_Gast_vs_Met_ordered, ]

# Adding cut-off for padj < 0.05
sign_highex_res_Gast_vs_Met <- highex_res_Gast_vs_Met_ordered_decreasing[highex_res_Gast_vs_Met_ordered_decreasing$padj <= 0.05, ]

#write.csv(sign_highex_res_Gast_vs_Met, "../DESeq-Sofia/significant_DEseq_results_Gast_vs_Met.csv", row.names = TRUE)
# Display the significant results
sign_highex_res_Gast_vs_Met


log2 fold change (MLE): library_name Gast vs Met 
Wald test p-value: library_name Gast vs Met 
DataFrame with 0 rows and 6 columns


In [21]:
%%R
#Now compare the first cancer stage to the second to last (Gast vs EA)

highex_res_Gast_vs_EA <- results(highex_dds, contrast = c("library_name", "Gast", "EA"))
clean_res_Gast_vs_EA <- highex_res_Gast_vs_EA[!is.na(highex_res_Gast_vs_EA$padj), ]

# Ordering by adjusted p-value (padj)
highex_res_Gast_vs_EA_ordered <- order(clean_res_Gast_vs_EA$padj, decreasing = FALSE)
highex_res_Gast_vs_EA_ordered_decreasing <- clean_res_Gast_vs_EA[highex_res_Gast_vs_EA_ordered, ]

# Adding cut-off for padj < 0.05
sign_highex_res_Gast_vs_EA <- highex_res_Gast_vs_EA_ordered_decreasing[highex_res_Gast_vs_EA_ordered_decreasing$padj <= 0.05, ]

#write.csv(sign_highex_res_Gast_vs_Met, "../DESeq-Sofia/significant_DEseq_results_Gast_vs_Met.csv", row.names = TRUE)
# Display the significant results
sign_highex_res_Gast_vs_EA

log2 fold change (MLE): library_name Gast vs EA 
Wald test p-value: library_name Gast vs EA 
DataFrame with 0 rows and 6 columns


In [22]:
%%R
#Now compare the second cancer stage to the third

highex_res_Atr_vs_EA <- results(highex_dds, contrast = c("library_name", "Gast", "EA"))
clean_res_Atr_vs_EA <- highex_res_Atr_vs_EA[!is.na(highex_res_Atr_vs_EA$padj), ]

# Ordering by adjusted p-value (padj)
highex_res_Atr_vs_EA_ordered <- order(clean_res_Atr_vs_EA$padj, decreasing = FALSE)
highex_res_Atr_vs_EA_ordered_decreasing <- clean_res_Atr_vs_EA[highex_res_Atr_vs_EA_ordered, ]

# Adding cut-off for padj < 0.05
sign_highex_res_Atr_vs_EA <- highex_res_Atr_vs_EA_ordered_decreasing[highex_res_Atr_vs_EA_ordered_decreasing$padj <= 0.05, ]

#write.csv(sign_highex_res_Atr_vs_Met, "../DESeq-Sofia/significant_DEseq_results_Atr_vs_Met.csv", row.names = TRUE)
# Display the significant results
sign_highex_res_Atr_vs_EA


log2 fold change (MLE): library_name Gast vs EA 
Wald test p-value: library_name Gast vs EA 
DataFrame with 0 rows and 6 columns


In [23]:
%%R
highex_res_Atr_vs_EA_ordered_decreasing

log2 fold change (MLE): library_name Gast vs EA 
Wald test p-value: library_name Gast vs EA 
DataFrame with 850 rows and 6 columns
          baseMean log2FoldChange     lfcSE      stat    pvalue      padj
         <numeric>      <numeric> <numeric> <numeric> <numeric> <numeric>
HP_0573    1.11604       0.637008  1.285735  0.495442  0.620288  0.999589
HP_0020    1.53435      -0.135630  0.987915 -0.137289  0.890802  0.999589
HP_1325    3.53811       0.710449  0.797562  0.890776  0.373050  0.999589
HP_0828    1.65759      -1.214761  1.083980 -1.120649  0.262437  0.999589
HP_1237    7.03845      -1.329846  1.036065 -1.283555  0.199298  0.999589
...            ...            ...       ...       ...       ...       ...
HP_r01  3386.31645      -0.122129  0.314047 -0.388887  0.697360  0.999589
HP_r04  3383.35399      -0.132936  0.195339 -0.680540  0.496163  0.999589
HP_r06    58.43071       0.147343  0.682187  0.215986  0.828999  0.999589
HP_t07     7.39309      -0.523105  0.811889 -0.644306  

In [24]:
%%R
# what if we remove the samples from the cancer stages that did not have the bacteria
#rownames(highex_counts_DEseq)
#colnames(highex_counts_DEseq)

#From the genome-patient list given in the raw data we can see which samples had the bacteria (they sequenced the genome)

Hpylori_samples <- read.delim("../Data/raw/Genome-patient-list.txt", 
                              header = TRUE, 
                              sep = "\t", 
                              fill = TRUE,    # Fills missing values to prevent misalignment
                              quote = "",     # Avoids errors from unexpected quotes
                              stringsAsFactors = FALSE,
                              check.names = FALSE) # Prevents modification of column names
head(Hpylori_samples)
Hpylori_samples_names <- Hpylori_samples[,1]
Hpylori_samples_names <- Hpylori_samples_names[Hpylori_samples_names != ""]
Hpylori_samples_names #these are the samples we want to keep

 [1] "Gast1" "Gast2" "Gast3" "Gast4" "Gast5" "Gast6" "Atr1"  "Atr4"  "Atr5" 
[10] "Atr6"  "Atr7"  "Atr8"  "Atr9"  "EA1"   "EA2"   "EA3"   "EA4"   "EA5"  
[19] "Met1"  "Met3" 


In [25]:
%%R
#edit the metadata
coldata_DEseq_hpsamples <- read.table("../metadata_clean.txt", header= TRUE, sep=",", stringsAsFactors = FALSE) 
coldata_DEseq_hpsamples<- coldata_DEseq_hpsamples[coldata_DEseq_hpsamples $library_name %in% Hpylori_samples_names, ]
coldata_DEseq_hpsamples

   run_accession library_name
1      ERR950158         Atr1
3      ERR950160         Atr5
4      ERR950161         Atr7
5      ERR950162         Atr8
6      ERR950163         Atr9
7      ERR950164          EA2
8      ERR950165          EA3
9      ERR950166          EA5
12     ERR950169         Met3
14     ERR950171        Gast1
15     ERR950172        Gast2
16     ERR950173        Gast3
17     ERR950174        Gast4
18     ERR950175        Gast5
19     ERR950176        Gast6
24     ERR950181         Atr4
25     ERR950182         Atr6
26     ERR950183          EA1
27     ERR950184          EA4
28     ERR950185         Met1


In [26]:
%%R
coldata_DEseq_hpsamples$library_name <- gsub("\\d+", "", coldata_DEseq_hpsamples$library_name)
coldata_DEseq_hpsamples$library_name <- factor(coldata_DEseq_hpsamples$library_name, levels = c("Hp-", "Gast", "Atr", "EA", "Met"))

In [27]:
%%R
#edit the counts input
counts_DEseq_hpsamples <- read.table("../HP_counts_fixed_163.txt", header= TRUE, sep="\t", stringsAsFactors = FALSE ) 
counts_DEseq_hpsamples <- counts_DEseq_hpsamples[, colnames(counts_DEseq_hpsamples) %in% coldata_DEseq_hpsamples$run_accession]


In [28]:
%%R 
#now we can run the DE analysis again
hpsamples_dds <- DESeqDataSetFromMatrix(countData=counts_DEseq_hpsamples, 
                                      colData=coldata_DEseq_hpsamples , 
                                      design=~library_name) 
hpsamples_dds <- DESeq(hpsamples_dds)


R[write to console]: factor levels were dropped which had no samples

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing

R[write to console]: -- replacing outliers and refitting for 24 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)

R[write to console]: estimating dispersions

R[write to console]: fitting model and testing



In [29]:
%%R
#compare the first and last cancer stage
hpsamples_res_Gast_vs_Met <- results(hpsamples_dds, contrast = c("library_name", "Gast", "Met"))
clean_hpsamples_res_Gast_vs_Met <- na.omit(hpsamples_res_Gast_vs_Met)

# Ordering by adjusted p-value (padj)
hpsamples_res_Gast_vs_Met_ordered <- clean_hpsamples_res_Gast_vs_Met[order(clean_hpsamples_res_Gast_vs_Met$padj, decreasing = FALSE), ]

# Adding cut-off for padj < 0.05
sign_hpsamples_res_Gast_vs_Met <- hpsamples_res_Gast_vs_Met_ordered[hpsamples_res_Gast_vs_Met_ordered$padj <= 0.05, ]

# Display the significant results
sign_hpsamples_res_Gast_vs_Met

log2 fold change (MLE): library_name Gast vs Met 
Wald test p-value: library_name Gast vs Met 
DataFrame with 5 rows and 6 columns
         baseMean log2FoldChange     lfcSE      stat      pvalue        padj
        <numeric>      <numeric> <numeric> <numeric>   <numeric>   <numeric>
HP_1432  638.0910        5.35686  0.834609   6.41840 1.37713e-10 2.72671e-08
HP_0109   62.6569       -2.69104  0.663790  -4.05406 5.03372e-05 3.32225e-03
HP_0875  475.4773        1.36875  0.333849   4.09991 4.13307e-05 3.32225e-03
HP_0632   29.2539        3.07426  0.819261   3.75248 1.75097e-04 8.66731e-03
HP_0010  237.4027       -2.47771  0.695586  -3.56204 3.67983e-04 1.45721e-02


In [30]:
%%R
#compare the first and third cancer stage
hpsamples_res_Gast_vs_EA <- results(hpsamples_dds, contrast = c("library_name", "Gast", "EA"))
clean_hpsamples_res_Gast_vs_EA <- na.omit(hpsamples_res_Gast_vs_EA)

# Ordering by adjusted p-value (padj)
hpsamples_res_Gast_vs_EA_ordered <- clean_hpsamples_res_Gast_vs_EA[order(clean_hpsamples_res_Gast_vs_EA$padj, decreasing = FALSE), ]
hpsamples_res_Gast_vs_EA_ordered 
# Adding cut-off for padj < 0.05
sign_hpsamples_res_Gast_vs_EA <- hpsamples_res_Gast_vs_EA_ordered[hpsamples_res_Gast_vs_EA_ordered$padj <= 0.05, ]

# Display the significant results
sign_hpsamples_res_Gast_vs_EA  #no significant

log2 fold change (MLE): library_name Gast vs EA 
Wald test p-value: library_name Gast vs EA 
DataFrame with 0 rows and 6 columns


In [31]:
%%R
#compare the first and second cancer stage
hpsamples_res_Gast_vs_Atr <- results(hpsamples_dds, contrast = c("library_name", "Gast", "Atr"))
clean_hpsamples_res_Gast_vs_Atr <- na.omit(hpsamples_res_Gast_vs_Atr)

# Ordering by adjusted p-value (padj)
hpsamples_res_Gast_vs_Atr_ordered <- clean_hpsamples_res_Gast_vs_Atr[order(clean_hpsamples_res_Gast_vs_Atr$padj, decreasing = FALSE), ]
hpsamples_res_Gast_vs_Atr_ordered

#Adding cut-off for padj < 0.05

sign_hpsamples_res_Gast_vs_Atr <- hpsamples_res_Gast_vs_Atr_ordered[hpsamples_res_Gast_vs_Atr_ordered$padj <= 0.05, ]

# Display the significant results
sign_hpsamples_res_Gast_vs_Atr #no significant

log2 fold change (MLE): library_name Gast vs Atr 
Wald test p-value: library_name Gast vs Atr 
DataFrame with 0 rows and 6 columns


In [32]:
%%R
#compare the second and third
hpsamples_res_Atr_vs_EA <- results(hpsamples_dds, contrast = c("library_name", "Atr", "EA"))
clean_hpsamples_res_Atr_vs_EA <- na.omit(hpsamples_res_Atr_vs_EA)

# Ordering by adjusted p-value (padj)
hpsamples_res_Atr_vs_EA_ordered <- clean_hpsamples_res_Atr_vs_EA[order(clean_hpsamples_res_Atr_vs_EA$padj, decreasing = FALSE), ]
hpsamples_res_Atr_vs_EA_ordered
#Adding cut-off for padj < 0.05
sign_hpsamples_res_Atr_vs_EA <- hpsamples_res_Atr_vs_EA_ordered[hpsamples_res_Atr_vs_EA_ordered$padj <= 0.05, ]

#Display the significant results
sign_hpsamples_res_Atr_vs_EA #no significant (but HP_1200 padj  0.0624118)

log2 fold change (MLE): library_name Atr vs EA 
Wald test p-value: library_name Atr vs EA 
DataFrame with 0 rows and 6 columns


In [33]:
%%R
#compare the second and last cancer stage
hpsamples_res_Atr_vs_Met <- results(hpsamples_dds, contrast = c("library_name", "Atr", "Met"))
clean_hpsamples_res_Atr_vs_Met <- na.omit(hpsamples_res_Atr_vs_Met)

# Ordering by adjusted p-value (padj)
hpsamples_res_Atr_vs_Met_ordered <- clean_hpsamples_res_Atr_vs_Met[order(clean_hpsamples_res_Atr_vs_Met$padj, decreasing = FALSE), ]

# Adding cut-off for padj < 0.05
sign_hpsamples_res_Atr_vs_Met <- hpsamples_res_Atr_vs_Met_ordered[hpsamples_res_Atr_vs_Met_ordered$padj <= 0.05, ]

# Display the significant results
sign_hpsamples_res_Atr_vs_Met

log2 fold change (MLE): library_name Atr vs Met 
Wald test p-value: library_name Atr vs Met 
DataFrame with 3 rows and 6 columns
         baseMean log2FoldChange     lfcSE      stat      pvalue        padj
        <numeric>      <numeric> <numeric> <numeric>   <numeric>   <numeric>
HP_1432   638.091        5.28776  0.821727   6.43493 1.23531e-10 3.16239e-08
HP_1200    26.423       -2.31841  0.536234  -4.32350 1.53571e-05 1.96571e-03
HP_0875   475.477        1.24968  0.329032   3.79804 1.45842e-04 1.24452e-02


In [34]:
%%R
#compare the third and last cancer stage
hpsamples_res_EA_vs_Met <- results(hpsamples_dds, contrast = c("library_name", "EA", "Met"))
clean_hpsamples_res_EA_vs_Met <- na.omit(hpsamples_res_EA_vs_Met)

# Ordering by adjusted p-value (padj)
hpsamples_res_EA_vs_Met_ordered <- clean_hpsamples_res_EA_vs_Met[order(clean_hpsamples_res_EA_vs_Met$padj, decreasing = FALSE), ]

# Adding cut-off for padj < 0.05
sign_hpsamples_res_EA_vs_Met <- hpsamples_res_EA_vs_Met_ordered[hpsamples_res_EA_vs_Met_ordered$padj <= 0.05, ]

# Display the significant results
sign_hpsamples_res_EA_vs_Met

log2 fold change (MLE): library_name EA vs Met 
Wald test p-value: library_name EA vs Met 
DataFrame with 1 row and 6 columns
         baseMean log2FoldChange     lfcSE      stat      pvalue       padj
        <numeric>      <numeric> <numeric> <numeric>   <numeric>  <numeric>
HP_1432   638.091        3.98901  0.852728   4.67794 2.89774e-06 0.00437559


In [35]:
%%R
#Do the same analysis with the removed low expressed genes
#only need to change the count file

counts_DEseq_hpsamples_highex <- read.csv("../HP_counts_fixed_163_highexpression_withrows", header= TRUE, sep=",", stringsAsFactors = FALSE, row.names=1 ) 
counts_DEseq_hpsamples_highex <- counts_DEseq_hpsamples_highex[, colnames(counts_DEseq_hpsamples_highex) %in% coldata_DEseq_hpsamples$run_accession]

nrow(counts_DEseq_hpsamples_highex)
ncol(counts_DEseq_hpsamples_highex)

[1] 20


In [36]:
%%R 
#now we can run the DE analysis again
hpsamples_highex_dds <- DESeqDataSetFromMatrix(countData=counts_DEseq_hpsamples_highex, 
                                      colData=coldata_DEseq_hpsamples , 
                                      design=~library_name) 
hpsamples_highex_dds <- DESeq(hpsamples_highex_dds)


R[write to console]: factor levels were dropped which had no samples

R[write to console]: estimating size factors

R[write to console]: estimating dispersions

R[write to console]: gene-wise dispersion estimates

R[write to console]: mean-dispersion relationship

R[write to console]: final dispersion estimates

R[write to console]: fitting model and testing

R[write to console]: -- replacing outliers and refitting for 5 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)

R[write to console]: estimating dispersions

R[write to console]: fitting model and testing



In [37]:
%%R

#compare the first and last cancer stage
hpsamples_highex_res_Gast_vs_Met <- results(hpsamples_highex_dds, contrast = c("library_name", "Gast", "Met"))
clean_hpsamples_highex_res_Gast_vs_Met <- na.omit(hpsamples_highex_res_Gast_vs_Met)

# Ordering by adjusted p-value (padj)
hpsamples_highex_res_Gast_vs_Met_ordered <- clean_hpsamples_highex_res_Gast_vs_Met[order(clean_hpsamples_highex_res_Gast_vs_Met$padj, decreasing = FALSE), ]

# Adding cut-off for padj < 0.05
sign_hpsamples_highex_res_Gast_vs_Met <- hpsamples_highex_res_Gast_vs_Met_ordered[hpsamples_highex_res_Gast_vs_Met_ordered$padj <= 0.05, ]

write.csv(sign_hpsamples_highex_res_Gast_vs_Met, "../DESeq-Sofia/sign_hpsamples_highex_res_Gast_vs_Met.csv", row.names = TRUE)

# Display the significant results
sign_hpsamples_highex_res_Gast_vs_Met

log2 fold change (MLE): library_name Gast vs Met 
Wald test p-value: library_name Gast vs Met 
DataFrame with 5 rows and 6 columns
         baseMean log2FoldChange     lfcSE      stat      pvalue        padj
        <numeric>      <numeric> <numeric> <numeric>   <numeric>   <numeric>
HP_1432  638.0910        5.35795  0.846187   6.33187 2.42203e-10 4.65030e-08
HP_0109   62.6569       -2.69046  0.677096  -3.97352 7.08182e-05 6.79855e-03
HP_0632   29.2539        3.07970  0.829382   3.71325 2.04618e-04 9.82166e-03
HP_0875  475.4773        1.36588  0.361287   3.78060 1.56452e-04 9.82166e-03
HP_0010  237.4027       -2.47689  0.709248  -3.49228 4.78915e-04 1.83903e-02


In [38]:
%%R
#compare the first and third cancer stage
hpsamples_highex_res_Gast_vs_EA <- results(hpsamples_highex_dds, contrast = c("library_name", "Gast", "EA"))
clean_hpsamples_highex_res_Gast_vs_EA <- na.omit(hpsamples_highex_res_Gast_vs_EA)

# Ordering by adjusted p-value (padj)
hpsamples_highex_res_Gast_vs_EA_ordered <- clean_hpsamples_highex_res_Gast_vs_EA[order(clean_hpsamples_highex_res_Gast_vs_EA$padj, decreasing = FALSE), ]

# Adding cut-off for padj < 0.05
sign_hpsamples_highex_res_Gast_vs_EA <- hpsamples_highex_res_Gast_vs_EA_ordered[hpsamples_highex_res_Gast_vs_EA_ordered$padj <= 0.05, ]

# Display the significant results
sign_hpsamples_highex_res_Gast_vs_EA

log2 fold change (MLE): library_name Gast vs EA 
Wald test p-value: library_name Gast vs EA 
DataFrame with 0 rows and 6 columns


In [39]:
%%R
#compare the first and second cancer stage
hpsamples_highex_res_Gast_vs_Atr <- results(hpsamples_highex_dds, contrast = c("library_name", "Gast", "Atr"))
clean_hpsamples_highex_res_Gast_vs_Atr <- na.omit(hpsamples_highex_res_Gast_vs_Atr)

# Ordering by adjusted p-value (padj)
hpsamples_highex_res_Gast_vs_Atr_ordered <- clean_hpsamples_highex_res_Gast_vs_Atr[order(clean_hpsamples_highex_res_Gast_vs_Atr$padj, decreasing = FALSE), ]

# Adding cut-off for padj < 0.05
sign_hpsamples_highex_res_Gast_vs_Atr <- hpsamples_highex_res_Gast_vs_Atr_ordered[hpsamples_highex_res_Gast_vs_Atr_ordered$padj <= 0.05, ]

# Display the significant results
sign_hpsamples_highex_res_Gast_vs_Atr

log2 fold change (MLE): library_name Gast vs Atr 
Wald test p-value: library_name Gast vs Atr 
DataFrame with 0 rows and 6 columns


In [40]:
%%R
#compare the second and third cancer stage
hpsamples_highex_res_Atr_vs_EA <- results(hpsamples_highex_dds, contrast = c("library_name", "Atr", "EA"))
clean_hpsamples_highex_res_Atr_vs_EA <- na.omit(hpsamples_highex_res_Atr_vs_EA)

# Ordering by adjusted p-value (padj)
hpsamples_highex_res_Atr_vs_EA_ordered <- clean_hpsamples_highex_res_Atr_vs_EA[order(clean_hpsamples_highex_res_Atr_vs_EA$padj, decreasing = FALSE), ]

# Adding cut-off for padj < 0.05
sign_hpsamples_highex_res_Atr_vs_EA <- hpsamples_highex_res_Atr_vs_EA_ordered[hpsamples_highex_res_Atr_vs_EA_ordered$padj <= 0.05, ]

# Display the significant results
sign_hpsamples_highex_res_Atr_vs_EA

log2 fold change (MLE): library_name Atr vs EA 
Wald test p-value: library_name Atr vs EA 
DataFrame with 0 rows and 6 columns


In [41]:
%%R
#compare the second and fourth cancer stage
hpsamples_highex_res_Atr_vs_Met <- results(hpsamples_highex_dds, contrast = c("library_name", "Atr", "Met"))
clean_hpsamples_highex_res_Atr_vs_Met <- na.omit(hpsamples_highex_res_Atr_vs_Met)

# Ordering by adjusted p-value (padj)
hpsamples_highex_res_Atr_vs_Met_ordered <- clean_hpsamples_highex_res_Atr_vs_Met[order(clean_hpsamples_highex_res_Atr_vs_Met$padj, decreasing = FALSE), ]

# Adding cut-off for padj < 0.05
sign_hpsamples_highex_res_Atr_vs_Met <- hpsamples_highex_res_Atr_vs_Met_ordered[hpsamples_highex_res_Atr_vs_Met_ordered$padj <= 0.05, ]

write.csv(sign_hpsamples_highex_res_Atr_vs_Met, "../DESeq-Sofia/sign_hpsamples_highex_res_Atr_vs_Met.csv", row.names = TRUE)

# Display the significant results
sign_hpsamples_highex_res_Atr_vs_Met

log2 fold change (MLE): library_name Atr vs Met 
Wald test p-value: library_name Atr vs Met 
DataFrame with 3 rows and 6 columns
         baseMean log2FoldChange     lfcSE      stat      pvalue        padj
        <numeric>      <numeric> <numeric> <numeric>   <numeric>   <numeric>
HP_1432   638.091        5.28878  0.833078   6.34849 2.17445e-10 5.24043e-08
HP_1200    26.423       -2.31460  0.553375  -4.18269 2.88078e-05 3.47134e-03
HP_0875   475.477        1.24587  0.355900   3.50060 4.64207e-04 3.72913e-02


In [42]:
%%R
#compare the third and fourth cancer stage
hpsamples_highex_res_EA_vs_Met <- results(hpsamples_highex_dds, contrast = c("library_name", "EA", "Met"))
clean_hpsamples_highex_res_EA_vs_Met <- na.omit(hpsamples_highex_res_EA_vs_Met)

# Ordering by adjusted p-value (padj)
hpsamples_highex_res_EA_vs_Met_ordered <- clean_hpsamples_highex_res_EA_vs_Met[order(clean_hpsamples_highex_res_EA_vs_Met$padj, decreasing = FALSE), ]

# Adding cut-off for padj < 0.05
sign_hpsamples_highex_res_EA_vs_Met <- hpsamples_highex_res_EA_vs_Met_ordered[hpsamples_highex_res_EA_vs_Met_ordered$padj <= 0.05, ]

write.csv(sign_hpsamples_highex_res_EA_vs_Met, "../DESeq-Sofia/sign_hpsamples_highex_res_EA_vs_Met.csv", row.names = TRUE)

# Display the significant results
sign_hpsamples_highex_res_EA_vs_Met

log2 fold change (MLE): library_name EA vs Met 
Wald test p-value: library_name EA vs Met 
DataFrame with 1 row and 6 columns
         baseMean log2FoldChange     lfcSE      stat      pvalue       padj
        <numeric>      <numeric> <numeric> <numeric>   <numeric>  <numeric>
HP_1432   638.091        3.99014  0.864613   4.61494 3.93214e-06 0.00335018


In [44]:
%%R
#total results
hpsamples_highex_res <- results(hpsamples_highex_dds)